In [1]:
import os
import time
import torch
import torchvision
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import cv2
import yaml
from PIL import Image
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import nms

# Configuração do dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Usando dispositivo: {device}")

# Carregar configurações do dataset
print("📂 Carregando configurações do dataset...")
with open("dataset.yaml", "r") as file:
    dataset_config = yaml.safe_load(file)

train_dir = dataset_config["train"]
val_dir = dataset_config["val"]
nc = dataset_config["nc"]
class_names = dataset_config["names"]
print("✅ Configurações carregadas com sucesso!")

# Definição da Backbone (CNN)
class Backbone(nn.Module):
    def __init__(self):
        super(Backbone, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))
        return x

# Definição da RPN
class RPN(nn.Module):
    def __init__(self, in_channels=512, num_anchors=9):
        super(RPN, self).__init__()
        self.conv = nn.Conv2d(in_channels, 512, kernel_size=3, padding=1)
        self.cls_logits = nn.Conv2d(512, num_anchors * 2, kernel_size=1)
        self.bbox_pred = nn.Conv2d(512, num_anchors * 4, kernel_size=1)

    def forward(self, x):
        x = F.relu(self.conv(x))
        logits = self.cls_logits(x)
        bbox_reg = self.bbox_pred(x)
        return logits, bbox_reg

# ROI Pooling
class ROIPooling(nn.Module):
    def __init__(self, output_size=(7, 7)):
        super(ROIPooling, self).__init__()
        self.output_size = output_size

    def forward(self, x, proposals):
        pooled_regions = [F.adaptive_max_pool2d(x[..., p[1]:p[3], p[0]:p[2]], self.output_size) for p in proposals]
        return torch.cat(pooled_regions, 0)

# Cabeça de detecção
class DetectionHead(nn.Module):
    def __init__(self, num_classes):
        super(DetectionHead, self).__init__()
        self.fc1 = nn.Linear(512 * 7 * 7, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.cls_score = nn.Linear(512, num_classes)
        self.bbox_pred = nn.Linear(512, num_classes * 4)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        class_logits = self.cls_score(x)
        bbox_regression = self.bbox_pred(x)
        return class_logits, bbox_regression

# Modelo Faster R-CNN
class FasterRCNN(nn.Module):
    def __init__(self, num_classes):
        super(FasterRCNN, self).__init__()
        self.backbone = Backbone()
        self.rpn = RPN()
        self.roi_pooling = ROIPooling()
        self.head = DetectionHead(num_classes)

    def forward(self, images):
        features = self.backbone(images)
        rpn_logits, rpn_bbox = self.rpn(features)
        proposals = torch.rand((5, 4))  # Placeholder para propostas de região
        roi_features = self.roi_pooling(features, proposals)
        class_logits, bbox_regression = self.head(roi_features)
        return class_logits, bbox_regression

# Criar Dataset e DataLoaders
class ObjectDetectionDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_filenames = sorted([f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

transform = transforms.Compose([transforms.ToTensor()])
train_dataset = ObjectDetectionDataset(train_dir, transform=transform)
val_dataset = ObjectDetectionDataset(val_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

# Treinamento
torch.manual_seed(42)
model = FasterRCNN(num_classes=nc + 1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 5
print("🚀 Iniciando treinamento...")
model.train()

for epoch in range(num_epochs):
    total_loss = 0
    for batch_idx, images in enumerate(train_loader):
        images = images.to(device)
        optimizer.zero_grad()
        class_logits, bbox_regression = model(images)
        loss = torch.mean(class_logits) + torch.mean(bbox_regression)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        print(f"Batch {batch_idx + 1}/{len(train_loader)} - Loss: {loss.item():.4f}")
    print(f"Época {epoch + 1} finalizada. Loss Total: {total_loss:.4f}")

torch.save(model.state_dict(), "faster_rcnn_scratch.pth")
print("💾 Modelo salvo com sucesso!")




# Função para testar uma imagem
def test_image(image_path, confidence_threshold=0.6):  # Ajustamos o threshold para evitar falsas detecções
    model.eval()
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model(image_tensor)

    image = cv2.imread(image_path)
    for i, box in enumerate(prediction[0]["boxes"]):
        confidence = prediction[0]["scores"][i].cpu().numpy()
        if confidence > confidence_threshold:
            x_min, y_min, x_max, y_max = box.cpu().numpy().astype(int)
            label = int(prediction[0]["labels"][i].cpu().numpy())

            cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
            cv2.putText(image, f"{class_names[label]}: {confidence:.2f}", (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX,
                        0.5, (0, 255, 0), 2)

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()


# Teste rápido do modelo
print("🖼️ Testando algumas imagens...")
valid_images = sorted([f for f in os.listdir(val_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])[:3]

if not valid_images:
    print("⚠️ Nenhuma imagem válida encontrada para teste!")
else:
    for img_name in valid_images:
        print(f"🟡 Testando imagem: {img_name}")
        test_image(os.path.join(val_dir, img_name), confidence_threshold=0.6)

print("✅ Teste finalizado!")



# Teste rápido do modelo
print("🖼️ Testando algumas imagens...")
valid_images = sorted([f for f in os.listdir(val_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])

if not valid_images:
    print("⚠️ Nenhuma imagem válida encontrada para teste!")
else:
    for img_name in valid_images:
        print(f"🟡 Testando imagem: {img_name}")
        test_image(os.path.join(val_dir, img_name), confidence_threshold=0.6)

print("✅ Teste finalizado!")


🖥️ Usando dispositivo: cpu
📂 Carregando configurações do dataset...
✅ Configurações carregadas com sucesso!
🚀 Iniciando treinamento...


RuntimeError: stack expects each tensor to be equal size, but got [3, 480, 640] at entry 0 and [3, 400, 300] at entry 1